# A2 - Knowledge-Base Demo
Show OCR quality on a sample and one working retrieval example.

## Retrieval demo
Run a natural Bangla question against the seeded FAISS index, print the retrieved chunk details, and assert that the expected page is retrieved.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

from doc_agent import config
from doc_agent.index import embed, store

query = "দলিলে টেনু সাব্বির পিতার নাম কী?"
expected_page_id = "dolil_605"
k = 3

cfg = config.load(ROOT / "configs" / "config.yaml")
loaded = store.load(cfg)
query_vector = embed.encode_query(query, cfg)
scores, indices = loaded["index"].search(query_vector, k)

top_idx = int(indices[0][0])
top_score = float(scores[0][0])
top_record = loaded["metadata"][top_idx]

print(f"query: {query}")
print(f"expected_page_id: {expected_page_id}")
print(f"retrieved_page_id: {top_record['page_id']}")
print(f"chunk_id: {top_record['chunk_id']}")
print(f"score: {top_score:.6f}")
print("chunk:")
print(top_record["chunk_text"])

assert top_record["page_id"] == expected_page_id
